# Track 2 — Intelligent Well Archive: full pipeline

config check → build or load the evidence store → route → answer → verify → write `predictions.csv` → validate → cost summary → review queue.

**Offline by default.** Answers are replayed from the committed cache `indexes/llm_cache.json`: no API key, no network, deterministic.
Set `LIVE = True` only to regenerate the cache (needs an API key in the `OpenRouter_Key` environment variable and `LLM_MODEL_*` in `.env`).

In [ ]:
import os, sys
from pathlib import Path

here = Path.cwd()
sys.path.insert(0, str(here if (here / "archive_qa").is_dir() else here / "submission"))

import pandas as pd
from archive_qa import answer, config, llm, route, store, write
from archive_qa.locate import locate_stats

pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 100)

LIVE = False                      # offline replay from the committed cache (the reviewers' mode)
# Optional subset for smoke runs, e.g. NAPE_QUESTION_IDS="T2-016,T2-001". Default: all 32 questions.
QUESTION_IDS = [q for q in os.environ.get("NAPE_QUESTION_IDS", "").split(",") if q] or None

## 1. Config check

In [ ]:
status = config.check()
print(status)
assert status["DATA_ROOT"], status.get("error")
assert status["xml_reports"] == 32 and status["pdf_reports"] == 32 and status["questions_csv"]
llm_cfg = llm.load_config(live=LIVE)
print("LLM:", llm_cfg.describe())       # names and flags only; the key is never shown
print("variables in .env:", llm.env_names())

## 2. Evidence store (build, or load if already built)

In [ ]:
records = store.build_store(verbose=False) if not config.RECORDS_PATH.exists() else store.load_store()
reports = route.discover_reports(config.XML_DIR)
print(f"{len(records)} records, {len(reports)} reports")
stats = locate_stats(records, show_unlocated=False)

## 3. Route all 32 questions

In [ ]:
questions = route.load_questions()
routing = route.route_all(questions, reports)
route.save_routing_report(routing)
print(routing["mode"].value_counts().to_string(), "| early abstentions:", int(routing.early_abstain.sum()))

## 4. Answer and verify
Two cached Haiku runs per question (the second only feeds a disagreement flag), plus one escalation call to Sonnet only if the verifier fails or the model abstains although a matching evidence line exists. Early abstentions make no LLM call.

In [ ]:
client = llm.LLM.from_env(live=LIVE)
answers = answer.answer_all(questions, records, reports, client, ids=QUESTION_IDS)
summary = pd.DataFrame([{"question_id": a.question_id, "status": a.status, "mode": a.mode, "confidence": a.confidence,
                         "verified": a.verified, "calls": a.llm_calls, "early_abstain": a.early_abstain,
                         "cited": len(a.cited_ids)} for a in answers])
print(summary["status"].value_counts().to_string())
summary

## 5. Write `predictions.csv`, validate with the organiser's checker

In [ ]:
partial = QUESTION_IDS is not None
pred_path = write.write_predictions(answers, records, allow_partial=partial)
write.save_answers_json(answers, records)
report = write.run_validator()
print(pred_path.name, "rows:", len(pd.read_csv(pred_path)))
print("validator:", None if report is None else {"valid": report["valid"], "errors": report["structure"]["errors"] + report["csv"]["errors"],
                                                  "warnings": report["csv"]["warnings"]})
pd.read_csv(pred_path)[["question_id", "confidence", "answer", "evidence"]]

## 6. Cost summary

In [ ]:
cost_path = llm.write_cost_summary(client.cache, client)
print(f"live calls this run: {client.live_calls}, cache hits: {client.cache_hits}, spent this run: ${client.spent:.4f}")
pd.read_csv(cost_path).tail(8)

## 7. Human review queue
Every answer whose verifier failed, and every low-confidence answer (early abstentions are excluded: they cite what was checked).

In [ ]:
queue = [a for a in answers if a.needs_review]
print(f"{len(queue)} answer(s) need human review")
pd.DataFrame([{"question_id": a.question_id, "confidence": a.confidence, "status": a.status,
               "why": next((n for n in a.notes if n.startswith("VERIFIER FAILED")), "low confidence")} for a in queue])

## 8. Review pack (optional, writes images from the PDFs; git-ignored)

In [ ]:
BUILD_REVIEW = False
if BUILD_REVIEW:
    from eval import review
    print(review.build_review(records, questions))